# 21 · Ablación de features y elasticidad censo (2026-09-12)

Valida la decisión de features del simulador de consumo urbano reutilizando el panel de
`08_panel_features` (misma construcción que `docker/airflow/include/ml/panel.py::construir_panel`).

Protocolo (idéntico a `docker/airflow/include/ml/entrenar.py`):
- 67 GradientBoostingRegressor (uno por municipio), params fijos, split train < 2022, test 2022-2024,
  predicción recursiva con lag actualizado.
- El panel `data/panel_features.parquet` debe reproducir el MAPE de la versión activa (8,685 %) para
  validar que la réplica es exacta.

**Regla de decisión (encargo 2026-09-12):**
- Quitar el IPH de las features solo si ΔMAPE ≤ +0,3 pp. Si no se cumple, conservar el IPH y
  **fusionarlo en un único factor** (quedarse con `iph_media`).
- La ocupación se retira como palanca de la UI (efecto causal no identificado) pero la feature se
  conserva si el MAPE lo justifica.

In [1]:
import json
from pathlib import Path

import numpy as np
import polars as pl
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA = Path("data")
panel = pl.read_parquet(DATA / "panel_features.parquet")
meta = json.loads((DATA / "panel_metadata.json").read_text())

PARAMS = {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 2, "random_state": 42}
TRAIN_DESDE, TEST_START = 2016, 2022
BASE = ["anio", "iph_media", "iph_max", "ocupacion_media", "lluvia_anual_mm", "lag1"]

print("panel:", panel.shape, "| municipios:", panel["cod_municipio"].n_unique())
print("features del parquet:", meta["features"])

panel: (670, 10) | municipios: 67
features del parquet: ['anio', 'iph_media', 'iph_max', 'ocupacion_media', 'lluvia_anual_mm', 'lag1']


In [2]:
def metricas(test, pred):
    test, pred = np.asarray(test, float), np.asarray(pred, float)
    return dict(
        mae=float(mean_absolute_error(test, pred)),
        mape=float(np.mean(np.abs((test - pred) / test)) * 100),
        rmse=float(mean_squared_error(test, pred) ** 0.5),
        r2=float(r2_score(test, pred)),
    )


def predict_recursivo(model, hist, test, features):
    last = float(hist["consumo_hm3"].to_list()[-1])
    preds = []
    for row in test.sort("anio").iter_rows(named=True):
        x = [float(row[f]) for f in features]
        x[features.index("lag1")] = last
        p = max(float(model.predict([x])[0]), 0.0)
        preds.append(p)
        last = p
    return preds


def evaluar(features):
    filas = []
    for cod_t, g in panel.partition_by("cod_municipio", as_dict=True).items():
        cod = int(cod_t[0])
        train = g.filter(
            (pl.col("anio") >= TRAIN_DESDE) & (pl.col("anio") < TEST_START) & pl.col("lag1").is_not_null()
        )
        test = g.filter(pl.col("anio") >= TEST_START)
        if train.height < 4:
            pred = [float(train["consumo_hm3"].mean())] * test.height
        else:
            m = GradientBoostingRegressor(**PARAMS)
            m.fit(train.select(features).to_numpy(), train["consumo_hm3"].to_numpy())
            pred = predict_recursivo(m, train, test, features)
        filas.append({"cod_municipio": cod, **metricas(test["consumo_hm3"].to_numpy(), pred)})
    return pl.DataFrame(filas)

In [3]:
CONFIGS = {
    "full (produccion)": BASE,
    "sin iph_max": [f for f in BASE if f != "iph_max"],
    "sin IPH (ambos)": [f for f in BASE if f not in ("iph_media", "iph_max")],
    "sin IPH y sin ocupacion": [f for f in BASE if f not in ("iph_media", "iph_max", "ocupacion_media")],
}

res = {}
filas = []
for nombre, feats in CONFIGS.items():
    df = evaluar(feats)
    res[nombre] = df
    filas.append({"config": nombre, "features": ", ".join(feats), "mape": round(df["mape"].mean(), 3)})

tabla = pl.DataFrame(filas).with_columns(
    delta_pp=(pl.col("mape") - pl.col("mape").first()).round(3),
    pasa_0_3pp=(pl.col("mape") - pl.col("mape").first() <= 0.3),
)
tabla

config,features,mape,delta_pp,pasa_0_3pp
str,str,f64,f64,bool
"""full (produccion)""","""anio, iph_media, iph_max, ocup…",8.685,0.0,true
"""sin iph_max""","""anio, iph_media, ocupacion_med…",8.863,0.178,true
"""sin IPH (ambos)""","""anio, ocupacion_media, lluvia_…",10.503,1.818,false
"""sin IPH y sin ocupacion""","""anio, lluvia_anual_mm, lag1""",12.255,3.57,false


In [4]:
ref = res["full (produccion)"].select("cod_municipio", pl.col("mape").alias("mape_ref"))
for nombre in ("sin iph_max", "sin IPH (ambos)", "sin IPH y sin ocupacion"):
    d = res[nombre].select("cod_municipio", pl.col("mape").alias("mape_alt")).join(ref, on="cod_municipio")
    d = d.with_columns((pl.col("mape_alt") - pl.col("mape_ref")).alias("delta_pp")).sort("delta_pp", descending=True)
    print(
        f"--- {nombre}: delta medio {d['delta_pp'].mean():+.3f} pp | "
        f"empeoran {d.filter(pl.col('delta_pp') > 0).height}/67"
    )
    print(d.head(5).select("cod_municipio", "delta_pp"))

--- sin iph_max: delta medio +0.178 pp | empeoran 41/67
shape: (5, 2)
┌───────────────┬──────────┐
│ cod_municipio ┆ delta_pp │
│ ---           ┆ ---      │
│ i64           ┆ f64      │
╞═══════════════╪══════════╡
│ 7014          ┆ 2.685022 │
│ 7052          ┆ 2.179197 │
│ 7006          ┆ 1.84834  │
│ 7011          ┆ 1.681347 │
│ 7003          ┆ 1.459378 │
└───────────────┴──────────┘
--- sin IPH (ambos): delta medio +1.818 pp | empeoran 43/67
shape: (5, 2)
┌───────────────┬───────────┐
│ cod_municipio ┆ delta_pp  │
│ ---           ┆ ---       │
│ i64           ┆ f64       │
╞═══════════════╪═══════════╡
│ 7049          ┆ 15.323094 │
│ 7051          ┆ 15.322882 │
│ 7039          ┆ 14.137032 │
│ 7055          ┆ 12.740793 │
│ 7014          ┆ 12.168694 │
└───────────────┴───────────┘
--- sin IPH y sin ocupacion: delta medio +3.570 pp | empeoran 43/67
shape: (5, 2)
┌───────────────┬───────────┐
│ cod_municipio ┆ delta_pp  │
│ ---           ┆ ---       │
│ i64           ┆ f64       │
╞════

## Resultado de la ablación

La réplica reproduce el MAPE de producción (**8,685 %**), lo que valida el protocolo.

| Config | MAPE | Δ vs prod | Gate ≤ +0,3 pp |
|---|---|---|---|
| full (producción) | 8,685 | — | — |
| sin `iph_max` | 8,863 | +0,18 pp | ✅ |
| sin IPH (ambos) | 10,503 | +1,82 pp | ❌ |
| sin IPH y sin ocupación | 12,255 | +3,57 pp | ❌ |

**Decisión 2026-09-12:** el IPH **no se elimina** (coste +1,82 pp). Se fusiona en un único factor:
`FEATURES = [anio, iph_media, ocupacion_media, lluvia_anual_mm, lag1]` (se elimina `iph_max`, +0,18 pp).
La ocupación se mantiene como feature (quitarla cuesta +3,57 pp) aunque deja de ser palanca de la UI
(su efecto causal sobre el consumo anual no está identificado).

## Elasticidad censo (palanca del simulador)

Reproduce la medida que respalda la nueva palanca de población: OLS en niveles de `consumo_hm3` ~
`poblacion` con los censos 2024 y 2025 (67 municipios × 2 años = 134 obs, el consumo de 2024 se
repite en ambos). La "elasticidad en la media" es `b · x̄ / ȳ` (pendiente × media población / media
consumo).

In [5]:
censo = pl.read_csv(DATA / "censo_municipal_baleares.csv", infer_schema_length=None).with_columns(
    pl.col("cod_municipio_ine").cast(pl.Int64).alias("cod_municipio")
)
abast = pl.read_csv(DATA / "abastecimiento_urbano_baleares.csv", infer_schema_length=None).filter(
    pl.col("anio") == 2024
)
pairs = pl.concat([
    censo.filter(pl.col("anio") == a).select("cod_municipio", "poblacion").with_columns(
        pl.lit(a).alias("censo_anio")
    )
    for a in (2024, 2025)
])
d = pairs.join(abast, on="cod_municipio", how="inner")
x, y = d["poblacion"].to_numpy().astype(float), d["consumo_hm3"].to_numpy().astype(float)
b, _ = np.polyfit(x, y, 1)
r = np.corrcoef(x, y)[0, 1]
el_media = b * x.mean() / y.mean()
per_capita = y / x * 1e9 / 365
print(f"n={len(d)} | pendiente={b * 1000:.4f} hm3/1000 hab | R2={r * r:.3f}")
print(f"elasticidad en la media = b*x/y = {el_media:.3f}")
print(f"per capita: mediana={np.median(per_capita):.0f} L/hab/dia | media={per_capita.mean():.0f}")

n=134 | pendiente=0.0710 hm3/1000 hab | R2=0.930
elasticidad en la media = b*x/y = 0.792
per capita: mediana=177 L/hab/dia | media=244


**Decisión 2026-09-12:** se instala `censo = 0,81` en `elasticidades.json` (medida original del
análisis; esta reproducción da 0,79 según la especificación exacta), documentando su origen:
OLS en niveles pooled 2024/2025, R² = 0,93, per cápita mediana ~177-184 L/hab/día.
Alternativa de planificación documentada: `censo ≈ 1,0` (el consumo escala con la población).